# Lecture 13 (Exceptions and file input/output)

## Exercise 13.1 (sum of two integers)

Write a program that as input takes a single line with two integers and outputs their sum. A valid input line consist of two integers separated by one or more white spaces. Input can be preceded and followed by white spaces. The program should repeat asking for input until a valid input is given.

    Input two integers, separated by space: 2 a
    invalid input
    Input two integers, separated by space: 2 3.0
    invalid input
    Input two integers, separated by space: 2 3
    Sum = 5

_Hint_. Use exceptions to handle illegal input.

In [ ]:
'''
Input a single line with two integers and output their sum.

Valid input are two integers separated by one or more whitespaces.
Input can be preceded and followed by white spaces.

Program repeats asking for input until a valid input is given.
'''

while True:
    try:
        text = input('Input two integers, separated by space: ')
#        x, y = [int(e) for e in text.split()]
        x, y = map(int,  text.split())
    except Exception:  # Exception to allow keyboard interrupt
        print('invalid input')
    else:
        break

print('Sum =', x + y)

## Exercise 13.2 (transposing)

In this exercise we assume that a file stores a matrix, where the rows are stored in consecutive lines in the file, and where column values are separated by `;`. Your task is to make a program that writes the [transposed matrix](https://en.wikipedia.org/wiki/Transpose) to another file - see example below.

Input:

    1;2;3;4
    5;6;7;8
    9;10;11;12

Output:

    1;5;9
    2;6;10
    3;7;11
    4;8;12

The program should ask the user for the name of an input file and the name of an output file, read the matrix in the input file, and write the transposed matrix to the output file.

Your solution should handle the following potential errors.

* If input file does not exist, report this to the user and start over.

* If output file already exists, warn the user, and ask the user to confirm to overwrite the existing file.

* If the lines in the input file have a different number of values, i.e. the input is not a valid matrix, report this to the user, and start over without writing any output.

_Hint_. Use the string method [`split`](https://docs.python.org/3/library/stdtypes.html#str.split) to split a line into a list of values.

In [ ]:
import os.path

FILE_IN = 'transpose_in.csv'
FILE_OUT = 'transpose_out.csv'

while True:
    filename_in = input('File to transpose [default ' + FILE_IN + ']: ')
    if not filename_in:
        filename_in = FILE_IN

    filename_out = input('Output file transpose [default ' + FILE_OUT + ']: ')
    if not filename_out:
        filename_out = FILE_OUT

    try:
        with open(filename_in) as file_in:
            lines = [line.strip().split(';') for line in file_in]
    except Exception:
        print('Error reading ', filename_in)
        continue

    if lines:
        columns = len(lines[0])
        if not all([len(line) == columns for line in lines]):
            print('Rows in file do not contain same number of columns')
            continue

    if os.path.isfile(filename_out):
        print(f'File {filename_out} already exist')
        overwrite = input('overwrite existing file ("yes" will overwrite): ')
        if overwrite.lower() != 'yes':
            continue

    try:
        with open(filename_out, 'w') as file_out:
            for row in zip(*lines):
                print(';'.join(row), file=file_out)
            print('done writting to', filename_out)
    except Exception:
        print('Failed writting output file', filename_out)
        continue
    break

## Exercise 13.3 (grade statistics)

In this exercise you should write a program that reads two files: one respectively containing the names and address of students, and one containing exam results. The output of the program should be a summary of the exam performances.

The student list is stored in a file, with one student per line, consisting of the three fields, separated by `;` : the unique student id, name and address.

    107;Donald Duck;Duck Steet 13, Duckburg
    243;Mickey Mouse;Mouse Street 42, Duckburg
    465;Goofy;Clumsy Road 7, Duckburg
    777;Scrooge McDuck;Money Street 1, Duckburg

The second file contains the list of exam results, consisting of a line per grade given, consisting of a triple, with values separated by `;` : student id, course name, and grade.

    107;Programming;10
    107;Mathematics;8
    107;Computability;7
    243;Programming;10
    243;Computer forensic;10
    243;Computability;9
    465;Mathematics;6

The program should print a summary like the below:

    Student id Name            Average #Courses
    ===========================================
    107        Donald Duck        8.33        3
    243        Mickey Mouse       9.67        3
    465        Goofy              6.00        1
    777        Scrooge McDuck     0.00        0

In [ ]:
STUDENTS = 'grades_students.csv'
EXAMS = 'grades_exams.csv'

names = {}
with open(STUDENTS) as student_file:
    for line in student_file:
        student_id, name, address = line.strip().split(';')
        names[student_id] = name

courses = {student_id: 0 for student_id in names}
grade_sum = {student_id: 0 for student_id in names}

with open(EXAMS) as exam_results:
    for line in exam_results:
        student_id, course, grade = line.strip().split(';')
        courses[student_id] += 1
        grade_sum[student_id] += int(grade)

name_width = max(map(len, names.values()))

header = f'Student id {"Name":{name_width}} Average #Courses'
print(header)
print('=' * len(header))
for student_id, name in sorted(names.items()):
    cnt = courses[student_id]
    average = grade_sum[student_id] / cnt if cnt else 0
    print(f'{student_id:10} {name:{name_width}} {average:7.2f} {cnt:8}')

## Exercise 13.4 (unicode lookup)

There exist many special Unicode characters, that can be used in Python. E.g., the following prints a snowman symbol (☃).

    print('\N{snowman}')

or

    import unicodedata
    print(unicodedata.lookup('snowman'))

In this exercise you should write a program that finds all Unicode names,
containing a substring. Your task is to download the file
[https://www.unicode.org/Public/15.0.0/ucd/UnicodeData.txt](https://www.unicode.org/Public/15.0.0/ucd/UnicodeData.txt),
that e.g. for the snowman symbol contains the line:

    2603;SNOWMAN;So;0;ON;;;;;N;;;;;

The first value is a hexadecimal (base 16) value representing the unicode number. To print a snowman, you can use `print(chr(int('2603', 16)))`.

Your program should ask for a substring, and print all symbol names and the symbol that contain the substring, e.g.,

    Unicode search: snowman
    SNOWMAN ☃
    SNOWMAN WITHOUT SNOW ⛄
    BLACK SNOWMAN ⛇

In [ ]:
symbols = {}
with open('UnicodeData.txt') as codes:
    for line in codes:
        hexcode, name, *_ = line.split(';')
        symbols[name] = int(hexcode, 16)

code = input('Unicode search: ').upper()
for symbol in symbols:
    if code in symbol:
        print(symbol, chr(symbols[symbol]))

## Exercise 13.5* (subset sum)

In this exercise we consider the [subset sum](https://en.wikipedia.org/wiki/Subset_sum_problem) problem.  Write a function `subset_sum(x, L)` that as input takes a value `x` and a list of values `L`.  The function should return a list `L_` with a subset `L` where `sum(L_) == x` if such a set `L_` exists, otherwise `None` should be returned.

_Example_.

    > print(subset_sum(12, [2, 3, 8, 11, -1]))
    [2, 11, -1]
    > print(subset_sum(6, [2, 3, 8, 11, -1]))
    None

The subset sum is known to be a _computationally hard_ problem that essentially only can be solved by considering all 2<sup>|`L`|</sup> possible subsets of the input list `L`.

_Hint_. Write a recursive function that generates all possible subsets, and raise an exception (ideally user defined)
during the recursion when the first solution has been found. Do not generate an explicit list of all subsets, since this will quickly lead to `MemoryError` exceptions.

In [ ]:
def subset_sum(x, L):
    class SolutionFound(Exception):
        pass


    def solve(s, i):
        '''Try to find a subset L' of L[i:] with sum s.

        solution contains already included elements from L[:i], i.e.
        sum(solution) + s == x.
        '''

        if s == 0:
            raise SolutionFound

        if i < len(L):
            solve(s, i + 1)  # skip L[i]
            solution.append(L[i])
            solve(s - L[i], i + 1)  # include L[i]
            solution.pop()

    try:
        solution = []
        solve(x, 0)
    except SolutionFound:
        return solution
    else:
        return None


print(subset_sum(12, [2, 3, 8, 11, -1]))
print(subset_sum(6, [2, 3, 8, 11, -1]))